# PrecisionForge Manufacturing EDA — Deep Dive
**The Darko Method 2026 | Module 06**

Answering four board-level questions:
- **Q1** — Does defect_rate_pct differ significantly by plant_type or shift?
- **Q2** — Which features most strongly predict defect_rate_pct?
- **Q3** — Does equipment_status correlate with defect severity?
- **Q4** — What is the forecast for defect_rate_pct next period?

In [ ]:
import sys, pathlib
root = pathlib.Path().resolve().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from scipy import stats

from src.eda_engine       import EDAEngine
from src.anomaly_detector import AnomalyDetector

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)

engine = EDAEngine()
(
    engine
    .load().profile().group_analysis().correlation()
    .anova_test().kruskal_test().feature_importance().forecast_next_period()
)
df = engine.df
print(f'Loaded {len(df):,} rows × {df.shape[1]} columns')
df.head(3)

---
## Chart 1 — Defect Rate Distribution
**Q1:** What proportion of runs breach the 5% board threshold?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(df['defect_rate_pct'], bins=50, kde=True, color='#3498db', ax=ax)
ax.axvline(5, color='#e74c3c', linewidth=2, linestyle='--', label='Board threshold (5%)')
over = (df['defect_rate_pct'] > 5).mean() * 100
ax.set_xlabel('Defect Rate (%)', fontsize=12)
ax.set_ylabel('Production Runs', fontsize=12)
ax.set_title(f'Q1 — Defect Rate Distribution  ({over:.1f}% of runs exceed 5%)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f'Mean defect rate:   {df["defect_rate_pct"].mean():.3f}%')
print(f'Median defect rate: {df["defect_rate_pct"].median():.3f}%')
print(f'Runs > 5%:          {(df["defect_rate_pct"] > 5).sum():,} ({over:.1f}%)')

---
## Chart 2 — Efficiency by Plant Type
**Q1:** Do plant types differ in efficiency_pct? Use a box plot to see spread, not just means.

In [ ]:
order = (
    df.groupby('plant_type')['efficiency_pct']
    .median().sort_values(ascending=False).index
)
fig, ax = plt.subplots(figsize=(11, 6))
sns.boxplot(data=df, x='plant_type', y='efficiency_pct', order=order,
            palette='Blues_r', ax=ax)
ax.axhline(df['efficiency_pct'].mean(), color='#e74c3c', linestyle='--', linewidth=1.5,
           label=f'Overall mean {df["efficiency_pct"].mean():.1f}%')
ax.set_xlabel('Plant Type', fontsize=12)
ax.set_ylabel('Efficiency (%)', fontsize=12)
ax.set_title('Q1 — Efficiency Distribution by Plant Type', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print('\nMedian efficiency by plant type:')
print(df.groupby('plant_type')['efficiency_pct'].median().sort_values(ascending=False).round(2).to_string())

---
## Chart 3 — Defect Type Bar Chart
**Q3:** Which defect types appear most frequently across all production runs?

In [ ]:
counts = df['primary_defect_type'].dropna().value_counts().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
palette = sns.color_palette('Reds_r', len(counts))
bars = ax.barh(counts.index, counts.values, color=palette, edgecolor='white')
for bar, val in zip(bars, counts.values):
    ax.text(val + 20, bar.get_y() + bar.get_height() / 2,
            f'{val:,}', va='center', fontsize=10)
ax.set_xlabel('Number of Runs', fontsize=12)
ax.set_title('Q3 — Defect Type Breakdown', fontsize=13, fontweight='bold')
ax.set_xlim(0, counts.max() * 1.15)
plt.tight_layout()
plt.show()

print('\nTop defect type:', counts.index[-1], f'({counts.iloc[-1]:,} runs)')

---
## Chart 4 — Time Trend of defect_rate_pct
**Q4:** Is the defect rate improving or worsening month over month?

In [ ]:
work = df.copy()
work['_month'] = pd.to_datetime(work['run_date'], errors='coerce').dt.to_period('M').astype(str)
monthly = (
    work.groupby('_month')['defect_rate_pct']
    .mean().reset_index().sort_values('_month')
)
monthly['rolling_3m'] = monthly['defect_rate_pct'].rolling(3).mean()

fc_result = engine.results.get('forecast', {}).get('defect_rate_pct', {})

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly['_month'], monthly['defect_rate_pct'],
        marker='o', markersize=4, linewidth=1.5, color='#3498db', label='Monthly mean')
ax.plot(monthly['_month'], monthly['rolling_3m'],
        linewidth=2.5, color='#2ecc71', linestyle='--', label='3-month rolling avg')
ax.axhline(5, color='#e74c3c', linestyle=':', linewidth=1.5, label='Board threshold (5%)')

n = max(1, len(monthly) // 8)
ax.set_xticks(monthly['_month'][::n])
ax.set_xticklabels(monthly['_month'][::n], rotation=45, ha='right')
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Mean Defect Rate (%)', fontsize=12)
ax.set_title('Q4 — Monthly Defect Rate Trend', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

if fc_result:
    print(f'Forecast (next period): {fc_result["next_period"]:.3f}%  — trend: {fc_result["trend"].upper()}')

---
## Chart 5 — Downtime vs Defect Rate
**Q3:** Does higher downtime correlate with higher defect rates? Coloured by equipment_status.

In [ ]:
sample = df.sample(min(3000, len(df)), random_state=42)
palette = {'Operational': '#3498db', 'Maintenance': '#e74c3c'}

fig, ax = plt.subplots(figsize=(9, 6))
for status, grp in sample.groupby('equipment_status'):
    ax.scatter(grp['downtime_mins'], grp['defect_rate_pct'],
               alpha=0.35, s=12, color=palette.get(status, '#95a5a6'), label=status)
ax.axhline(5, color='#e74c3c', linestyle='--', linewidth=1.2, alpha=0.7,
           label='5% threshold')
ax.set_xlabel('Downtime (mins)', fontsize=12)
ax.set_ylabel('Defect Rate (%)', fontsize=12)
ax.set_title('Q3 — Downtime vs Defect Rate by Equipment Status',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

r, p = stats.pearsonr(df['downtime_mins'].dropna(), df['defect_rate_pct'].dropna())
print(f'Pearson r (downtime vs defect_rate): {r:.4f}  (p={p:.4f})')

---
## Chart 6 — Severity Breakdown by Plant Type
**Q3:** Are Critical runs concentrated in specific plant types?

In [ ]:
order_sev = ['Critical', 'Major', 'Minor', 'None']
avail = [s for s in order_sev if s in df['primary_severity'].dropna().unique()]

pivot = (
    df.groupby(['plant_type', 'primary_severity'])
    .size().unstack(fill_value=0)
    .reindex(columns=avail, fill_value=0)
)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
pivot_pct = pivot_pct.sort_values('Critical', ascending=False)

colors = {'Critical': '#e74c3c', 'Major': '#f39c12', 'Minor': '#f1c40f', 'None': '#95a5a6'}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: stacked bar
bottom = np.zeros(len(pivot_pct))
for sev in avail:
    vals = pivot_pct[sev].values
    axes[0].bar(pivot_pct.index, vals, bottom=bottom,
                color=colors.get(sev, '#bdc3c7'), label=sev, edgecolor='white', width=0.6)
    bottom += vals
axes[0].set_ylabel('Share of Runs (%)', fontsize=12)
axes[0].set_xlabel('Plant Type', fontsize=12)
axes[0].set_title('Q3 — Severity Breakdown by Plant Type', fontsize=12, fontweight='bold')
axes[0].legend(title='Severity', fontsize=9)
axes[0].tick_params(axis='x', rotation=30)

# Right: Critical run mean defect by plant_type
crit = df[df['primary_severity'] == 'Critical']
crit_defect = (
    crit.groupby('plant_type')['defect_rate_pct']
    .mean().sort_values(ascending=True).round(3)
)
axes[1].barh(crit_defect.index, crit_defect.values,
             color=sns.color_palette('Reds_r', len(crit_defect)), edgecolor='white')
axes[1].set_xlabel('Mean Defect Rate on Critical Runs (%)', fontsize=12)
axes[1].set_title('Critical Run Defect Rate by Plant', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# ANOVA and Kruskal results
print('\nStatistical tests (defect_rate_pct by plant_type):')
for test in ['anova', 'kruskal']:
    res = engine.results.get(test, {}).get('defect_rate_pct ~ plant_type', {})
    if res:
        stat = res.get('f_stat') or res.get('h_stat')
        label = 'F' if test == 'anova' else 'H'
        sig = 'SIGNIFICANT' if res['significant'] else 'not significant'
        print(f'  {test.upper()}: {label}={stat:.3f}  p={res["p_value"]:.4f}  [{sig}]')

---
## Critical Runs Deep Dive
**Q3:** How much worse are Critical-severity runs compared to all others?

In [ ]:
detector = AnomalyDetector(df)
detector.run(columns=['defect_rate_pct', 'downtime_mins'])
detector.flag_critical_runs()

cr = detector.results['critical_runs']
print(f'Critical runs: {cr["critical_count"]:,} ({cr["critical_pct"]}% of all runs)\n')

for metric in ['defect_rate_pct', 'downtime_mins', 'efficiency_pct']:
    m = cr.get(metric, {})
    if m:
        print(f'{metric}')
        print(f'  Critical mean:     {m["critical_mean"]:.3f}')
        print(f'  Non-critical mean: {m["non_critical_mean"]:.3f}')
        print(f'  Delta:             {m["delta"]:+.3f}  ({m["ratio"]:.2f}×)')
        print()

---
## Summary of Findings

| Question | Finding |
|---|---|
| Q1 — Defect by plant | Precision and Chemical have highest rates; ANOVA not significant — differences are marginal |
| Q1 — Defect by shift | Afternoon slightly higher; statistically not significant |
| Q2 — Feature importance | All features show negligible correlation with defect_rate_pct (|r| < 0.02) |
| Q3 — Critical runs | Critical runs have 2.8× higher defect rate vs non-Critical (5.44% vs 1.94%) |
| Q4 — Forecast | Next period estimated at 2.41% — trend: WORSENING |

**Outputs generated:**
- `reports/analysis_report.txt` — full text EDA report
- `reports/manufacturing_eda_report.html` — styled HTML report
- `reports/anomalies.csv` — flagged production runs
- `reports/plots/` — 6 PNG charts